# dependencies

In [10]:
from typing import TypedDict, List, Dict, Any, Annotated, Union
from langchain_core.agents import AgentAction, AgentFinish
import operator
from langchain.agents import tool
from langchain_core.output_parsers import StrOutputParser
from datetime import datetime
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import TavilySearchResults
from langchain_core.prompts import PromptTemplate
from langgraph.graph import END, StateGraph

import sys
import logging
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(filename='logs/react_agent.log', mode='a')
    ]
)
logger = logging.getLogger(__name__)

from dotenv import load_dotenv
load_dotenv()

True

In [11]:
MODEL = "gemini-2.5-flash"
TEMPERATURE = 0.1
MAX_ITERATIONS = 5
MAX_LLM_RETRIES = 3 
SEARCH_DEPTH = 'basic'
MIN_REQUIRED_TOOL_STEPS = 2

# state definition

In [12]:
class AgentState(TypedDict):
    """Defines the structure of the agent's state throughout execution"""
    input: str
    agent_outcome: Union[AgentAction, AgentFinish, None]
    intermediate_steps: Annotated[List[tuple[AgentAction, str]], operator.add]
    iteration_count: int

# tools definition

In [13]:
@tool
def get_current_time(format_str: str = "%Y-%m-%d %H:%M:%S") -> str:
    """
    Returns the current date and time in the specified format
    
    Args:
        format_str: Python datetime format string
    
    Returns:
        Formatted current date and time
    """
    try:
        current_time = datetime.now().strftime(format_str)
        logger.info(f"✅ get_current_time executed successfully: {current_time}")
        return current_time
    except Exception as e:
        logger.error(f"❌ Error in get_current_time: {e}")
        return f"Error: {e}"
    
@tool
def calculate_days_between(
    date1: str = None,
    date2: str = None,
    start_date: str = None,
    end_date: str = None
) -> str:
    """
    Calculates the number of days between two dates.

    Accepted inputs (model may send any of these):
    - JSON: {"date1": "YYYY-MM-DD", "date2": "YYYY-MM-DD"}
    - JSON: {"start_date": "YYYY-MM-DD", "end_date": "YYYY-MM-DD"}
    - Positional (deprecated): date1, date2

    Returns:
        "<N> days"
    """
    try:
        # Normalize aliases
        if (not date1 or not date2) and start_date and end_date:
            date1, date2 = start_date, end_date
        if not date1 or not date2:
            return "❌ Error: both date1/date2 (or start_date/end_date) are required"

        d1 = datetime.strptime(date1, "%Y-%m-%d")
        d2 = datetime.strptime(date2, "%Y-%m-%d")
        delta = abs((d2 - d1).days)
        logger.info(f"✅ calculate_days_between executed successfully: {delta} days")
        return f"{delta} days"
    except Exception as e:
        logger.error(f"❌ Error in calculate_days_between: {e}")
        return f"Error: {e}"

# react agent class

In [14]:
REACT_PROMPT_TEMPLATE = PromptTemplate(
    input_variables=["input","tools","tool_names","agent_scratchpad","min_steps"], 
    template=
    """You are a reasoning + tool-using agent.

    TOOLS: {tools}

    REQUIREMENTS:

    Use at least {min_steps} tool calls (e.g. search, get_current_time, calculate_days_between) before Final Answer unless the question is purely definitional (this one is NOT).
    When using calculate_days_between, ALWAYS supply JSON exactly as: Action Input: {{"date1": "YYYY-MM-DD", "date2": "YYYY-MM-DD"}} (If you instead use start_date/end_date it is still accepted.)
    Output must strictly follow one of two schemas:
    Tool use step: Thought: <your reasoning> Action: <one of [{tool_names}]> Action Input: <plain text or JSON arg>

    Final answer (only after sufficient tool use): Thought: I now know the final answer Final Answer: <answer>

    Never skip Thought. Never invent tool names.

    Previous Steps: {agent_scratchpad}

    Question: {input} Provide the next step now."""
    )

In [15]:
class ReActAgent:
    def __init__(
        self,
        model: str = MODEL,
        temperature: float = TEMPERATURE,
        max_iterations: int = MAX_ITERATIONS,
        search_depth: str = SEARCH_DEPTH,
        require_tool_use: bool = True,
        min_required_tool_steps: int = MIN_REQUIRED_TOOL_STEPS,
        max_llm_retries: int = MAX_LLM_RETRIES,
    ):
        self.model = model
        self.temperature = temperature
        self.max_iterations = max_iterations
        self.search_depth = search_depth
        self.require_tool_use = require_tool_use
        self.min_required_tool_steps = min_required_tool_steps
        self.max_llm_retries = max_llm_retries

        self.llm = self._build_llm()
        self.tools = self._setup_tools()

        # Build a simple chain: Prompt -> LLM -> string
        self.react_chain = REACT_PROMPT_TEMPLATE | self.llm | StrOutputParser()

        self.graph = self._build_graph()
        logger.info("✅ ReActAgent initialized successfully")

    def _build_llm(self) -> ChatGoogleGenerativeAI:
        try:
            llm = ChatGoogleGenerativeAI(
                model=self.model,
                temperature=self.temperature,
            )
            logger.info(f"✅ Initialized LLM: {self.model} with temperature {self.temperature}")
            return llm
        except Exception as e:
            logger.error(f"❌ Error initializing LLM: {e}")
            raise

    def _setup_tools(self) -> List:
        try:
            search_tool = TavilySearchResults(depth=self.search_depth, max_results=5)
            tools = [get_current_time, calculate_days_between, search_tool]
            logger.info(f"✅ Set up {len(tools)} tools for the agent")
            return tools
        except Exception as e:
            logger.error(f"❌ Error setting up tools: {e}")
            raise

    # ------- Helper formatting / parsing -------

    def _format_tools(self) -> str:
        lines = []
        for t in self.tools:
            desc_full = (getattr(t, "description", "") or "").strip()
            first_line = desc_full.splitlines()[0] if desc_full else ""
            lines.append(f"{t.name}: {first_line}")
        logger.debug(f"Formatted tools (concise):\n{lines}")
        return "\n".join(lines)

    def _build_scratchpad(self, steps):
        if not steps:
            return "None\n"
        blocks = []
        for action, observation in steps:
            blocks.append(
                f"Thought: {action.log.split('Thought:',1)[1].strip().splitlines()[0] if 'Thought:' in action.log else ''}\n"
                f"Action: {action.tool}\nAction Input: {action.tool_input}\nObservation: {observation}"
            )
        
        logger.debug(f"Built scratchpad:\n{blocks}")
        return "\n\n".join(blocks) + "\n"

    def _parse_react_output(self, text: str):
        """
        Robust parse of model output.
        """
        import re
        try:
            raw = text or ""
            stripped = raw.strip()
            logger.debug(f"Parsing model output (len={len(raw)}): {repr(raw)}")
            if not stripped:
                return None  # signal empty -> caller handles retry
            if "Final Answer:" in raw:
                answer = raw.split("Final Answer:", 1)[1].strip()
                return AgentFinish(return_values={"output": answer}, log=raw)
            action_match = re.search(r"^Action:\s*(.+)$", raw, re.MULTILINE)
            input_match  = re.search(r"^Action Input:\s*(.+)$", raw, re.MULTILINE)
            if not action_match:
                # treat as provisional finish (may be re-prompted if premature)
                return AgentFinish(return_values={"output": stripped}, log=raw)
            tool_name = action_match.group(1).strip()
            tool_input = input_match.group(1).strip() if input_match else ""
            tool_input = tool_input.strip().strip('"').strip("'")
            return AgentAction(tool=tool_name, tool_input=tool_input, log=raw)
        except Exception as e:
            logger.error(f"❌ Parse failure: {e}")
            return AgentFinish(
                return_values={"output": f"Error: parse failure ({e})"},
                log=text
            )

    # --------------- Graph Nodes ----------------

    def _reason_node(self, state: AgentState) -> Dict[str, Any]:
        logger.info(f"🧠 Reasoning step {state.get('iteration_count', 0) + 1}")
        try:
            scratchpad = self._build_scratchpad(state["intermediate_steps"])
            prompt_inputs = {
                "input": state["input"],
                "tools": self._format_tools(),
                "tool_names": ", ".join([t.name for t in self.tools]),
                "agent_scratchpad": scratchpad,
                "min_steps": self.min_required_tool_steps
            }
            logger.debug(f"Prompt inputs: {prompt_inputs}")
            retries = 0
            outcome = None
            llm_output = ""
            while retries <= self.max_llm_retries:
                llm_output = self.react_chain.invoke(prompt_inputs).strip()
                if not llm_output:
                    retries += 1
                    logger.warning(f"⚠️ Empty LLM output (attempt {retries}/{self.max_llm_retries})")
                    prompt_inputs["agent_scratchpad"] += (
                        "Thought: The previous output was empty — I must produce either a valid Action or a Final Answer after required tool steps.\n"
                    )
                    if retries > self.max_llm_retries:
                        break
                    continue
                outcome = self._parse_react_output(llm_output)
                if outcome is None:  # parser signaled empty
                    retries += 1
                    continue
                # If it returned a Final Answer too early, force re-prompt
                if (self.require_tool_use
                    and isinstance(outcome, AgentFinish)
                    and len(state["intermediate_steps"]) < self.min_required_tool_steps):
                    retries += 1
                    logger.debug("🔁 Premature Final Answer; forcing additional tool reasoning.")
                    prompt_inputs["agent_scratchpad"] += (
                        f"Thought: I have only {len(state['intermediate_steps'])} tool step(s); "
                        f"I must use a tool next (choose from [{prompt_inputs['tool_names']}]).\n"
                    )
                    continue
                break
            if outcome is None:
                outcome = AgentFinish(
                    return_values={"output": "Error: empty model output after retries"},
                    log=llm_output
                )
            logger.debug(f"LLM output:\n{llm_output}")
            logger.debug(f"Parsed outcome: {outcome}")
            res = {
                "agent_outcome": outcome,
                "iteration_count": state.get("iteration_count", 0) + 1
            }
            logger.info("✅ Reasoning step completed")
            return res
        except Exception as e:
            logger.error(f"❌ Error in reasoning node: {e}")
            return {
                "agent_outcome": AgentFinish(
                    return_values={"output": f"Error: {e}"},
                    log=str(e)
                ),
                "iteration_count": state.get("iteration_count", 0) + 1
            }

    def _act_node(self, state: AgentState) -> Dict[str, Any]:
        action = state["agent_outcome"]
        if not isinstance(action, AgentAction):
            logger.warning("⚠️ No action to execute")
            return {"intermediate_steps": []}
        try:
            tool = next((t for t in self.tools if t.name == action.tool), None)
            if not tool:
                output = f"Tool '{action.tool}' not found"
            else:
                raw_input = action.tool_input
                parsed_input = raw_input
                # Attempt JSON parse for dict-based tools
                if isinstance(raw_input, str) and raw_input.strip().startswith("{") and raw_input.strip().endswith("}"):
                    import json
                    try:
                        parsed_input = json.loads(raw_input)
                        # Normalize keys for calculate_days_between
                        if tool.name == "calculate_days_between":
                            if "start_date" in parsed_input and "date1" not in parsed_input:
                                parsed_input["date1"] = parsed_input["start_date"]
                            if "end_date" in parsed_input and "date2" not in parsed_input:
                                parsed_input["date2"] = parsed_input["end_date"]
                    except Exception as je:
                        logger.warning(f"⚠️ JSON parse failed, using raw string: {je}")
                logger.debug(f"Invoking tool {tool.name} with normalized input: {parsed_input}")
                output = tool.invoke(parsed_input)
            logger.info(f"✅ Executed tool: {action.tool}")
            res = {"intermediate_steps": [(action, str(output))]}
            logger.debug(f"Act node returning state: {res}")
            return res
        except Exception as e:
            logger.error(f"❌ Tool execution error: {e}")
            return {"intermediate_steps": [(action, f"Error: {e}")]}

    def _should_continue(self, state: AgentState) -> str:
        count = state.get("iteration_count", 0)
        outcome = state["agent_outcome"]
        if count >= self.max_iterations:
            return END
        if isinstance(outcome, AgentFinish):
            return END
        if isinstance(outcome, AgentAction):
            return "act"
        return END

    def _build_graph(self):
        graph = StateGraph(AgentState)
        graph.add_node("reason", self._reason_node)
        graph.add_node("act", self._act_node)
        graph.set_entry_point("reason")
        graph.add_conditional_edges("reason", self._should_continue)
        graph.add_edge("act", "reason")
        logger.info("✅ State graph built")
        return graph.compile()

    # --------------- Public API -----------------

    def run(self, query: str) -> Dict[str, Any]:
        logger.info(f"🚀 Starting ReActAgent with question: {query}")
        init: AgentState = {
            "input": query,
            "agent_outcome": None,
            "intermediate_steps": [],
            "iteration_count": 0
        }
        final_state = self.graph.invoke(init)
        final_answer = ""
        if isinstance(final_state["agent_outcome"], AgentFinish):
            final_answer = final_state["agent_outcome"].return_values.get("output", "")
        tool_steps = len(final_state["intermediate_steps"])
        success = (
            isinstance(final_state["agent_outcome"], AgentFinish)
            and bool(final_answer.strip())
            and not final_answer.startswith("Error:")
            and (not self.require_tool_use or tool_steps >= self.min_required_tool_steps)
        )
        return {
            "question": query,
            "final_answer": final_answer,
            "intermediate_steps": final_state["intermediate_steps"],
            "iterations": final_state["iteration_count"],
            "success": success,
            "tool_steps": tool_steps
        }

    def visualize_graph(self):
        try:
            print("=== Mermaid Graph ===")
            print(self.graph.get_graph().draw_mermaid())
            print("=== ASCII Graph ===")
            print(self.graph.get_graph().draw_ascii())
        except Exception as e:
            print(f"Graph visualization error: {e}")

# main function

In [16]:
question = "When was SpaceX's last launch and how many days ago was that from this instant?"

In [17]:
def main():
    """Main function to demonstrate the ReactAgent"""
    try:
        logger.info("🎯 Initializing ReactAgent demo")

        # create agent instance
        agent = ReActAgent()

        result = agent.run(question)

        print(f"\n{'='*60}")
        print(f"QUESTION: {question}")
        print('='*60)
        print(f"\nFINAL ANSWER:")
        print(result["final_answer"])
        print(f"\nExecution Details:")
        print(f"- Iterations: {result['iterations']}")
        print(f"- Success: {result['success']}")
        print(f"- Steps taken: {len(result['intermediate_steps'])}")

        logger.info(f"QUESTION: {question}")
        logger.info(f"RESULT: {result}")

        # Show graph structure
        print(f"\n{'='*60}")
        agent.visualize_graph()

    except Exception as e:
        logger.error(f"❌ Error in main: {e}")
        print(f"Error in main: {e}")

In [18]:
main()

2025-08-26 11:34:23 | INFO     | __main__ | 🎯 Initializing ReactAgent demo
2025-08-26 11:34:23 | INFO     | __main__ | ✅ Initialized LLM: gemini-2.5-flash with temperature 0.1
2025-08-26 11:34:23 | INFO     | __main__ | ✅ Set up 3 tools for the agent
2025-08-26 11:34:23 | INFO     | __main__ | ✅ State graph built
2025-08-26 11:34:23 | INFO     | __main__ | ✅ ReActAgent initialized successfully
2025-08-26 11:34:23 | INFO     | __main__ | 🚀 Starting ReActAgent with question: When was SpaceX's last launch and how many days ago was that from this instant?
2025-08-26 11:34:23 | INFO     | __main__ | 🧠 Reasoning step 1
2025-08-26 11:34:23 | DEBUG    | __main__ | Formatted tools (concise):
['get_current_time: Returns the current date and time in the specified format', 'calculate_days_between: Calculates the number of days between two dates.', 'tavily_search_results_json: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions ab